In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, UnidentifiedImageError
from diffusers import UNet2DModel, DDPMScheduler
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt

train_df_full = pd.read_csv("/mnt/Internal/MedImage/chexpert_balanced_for_training_3000_per_label_dis+demog+age.csv")
image_root = "/mnt/Internal/MedImage/CheXpert Dataset/unzip_chexpert_images/CheXpert-v1.0/train"
# Filter rows where the 'view' column contains 'frontal'
train_df_full= train_df_full[train_df_full['Frontal/Lateral'].str.contains('frontal', case=False, na=False)]
# Data Preprocessing
train_df_full.replace(-1, 1, inplace=True)
train_df_full.replace(np.nan, 0, inplace=True)

# Validation Data
valid_df = train_df_full[175001:190000]
# Training Data
train_df = train_df_full[0:93000]

train_df_full = train_df_full



def generate_caption(row):
    findings = []

    for disease in [
        'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
        'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
        'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
    ]:
        if row[disease] == 1:
            findings.append(disease)

    if len(findings) == 0 or (len(findings) == 1 and 'No Finding' in findings):
        findings_caption = "no significant findings"
    else:
        findings_caption = ", ".join(findings).lower()

    gender = row.get("GENDER_Female", None)
    gender_str = "male" if gender == 0 else "female"

    age_group = next((col.replace("AGE_GROUP_", "").replace("_", " ").lower()
                      for col in row.index if "AGE_GROUP" in col and row[col] == 1), "unknown age")

    return f"Grayscale frontal chest X-ray radiograph of a {gender_str} patient aged {age_group}, showing {findings_caption}."


/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
ls "/mnt/Internal/MedImage/chexpert_balanced_for_training_3000_per_label_dis+demog+age.csv"

In [2]:
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError
import torch
import torchvision.transforms as transforms


In [3]:
train_df["caption"] = train_df.apply(generate_caption, axis=1)


/tmp/ipykernel_2092222/3789562243.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["caption"] = train_df.apply(generate_caption, axis=1)


In [4]:
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class CheXpertCaptionDataset(Dataset):
    def __init__(self, dataframe, image_root, tokenizer, image_size=512):
        self.dataframe = dataframe
        self.image_root = image_root
        self.tokenizer = tokenizer
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Ensure 3-channel normalization
        ])
        self.valid_data = []

        for idx, row in dataframe.iterrows():
            img_path = os.path.join(image_root, row["Path"].replace("CheXpert-v1.0/train/", ""))
            if os.path.exists(img_path):
                # Update caption with grayscale hint
                caption = self._update_caption_with_grayscale(row["caption"])
                self.valid_data.append((img_path, caption))

    def __len__(self):
        return len(self.valid_data)

    def __getitem__(self, idx):
        img_path, caption = self.valid_data[idx]

        # Load grayscale image and replicate to 3-channel RGB
        image = Image.open(img_path).convert("L").convert("RGB")
        image = self.transform(image)

        tokenized = self.tokenizer(caption, padding="max_length", truncation=True,
                                   max_length=77, return_tensors="pt")

        return {
            "pixel_values": image,
            "input_ids": tokenized.input_ids.squeeze(0),
            "attention_mask": tokenized.attention_mask.squeeze(0)
        }

    def _update_caption_with_grayscale(self, caption):
        """
        Adds 'grayscale frontal chest X-ray radiograph' phrase to the start of the caption.
        """
        if "grayscale" not in caption.lower():
            return "Grayscale frontal chest X-ray radiograph. " + caption
        return caption


In [5]:
import accelerate
print(accelerate.__version__)


1.6.0


In [6]:
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from transformers import CLIPTokenizer, CLIPTextModel

# Correct tokenizer and text encoder path
clip_model_id = "openai/clip-vit-large-patch14"
sd_model_id = "runwayml/stable-diffusion-v1-5"

# Load tokenizer and text encoder from CLIP
tokenizer = CLIPTokenizer.from_pretrained(clip_model_id)
text_encoder = CLIPTextModel.from_pretrained(clip_model_id)

# Load VAE and U-Net from Stable Diffusion
vae = AutoencoderKL.from_pretrained(sd_model_id, subfolder="vae")
unet = UNet2DConditionModel.from_pretrained(sd_model_id, subfolder="unet")

# Freeze everything except U-Net
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.train()


UNet2DConditionModel(
  (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): Linear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): Linear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): CrossAttnDownBlock2D(
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(
          (norm): GroupNorm(32, 320, eps=1e-06, affine=True)
          (proj_in): Conv2d(320, 320, kernel_size=(1, 1), stride=(1, 1))
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): Linear(in_features=320, out_features=320, bias=False)
                (to_k): Linear(in_features=320, out_features=320, bias=False)
                (to_v): Linear(in_features=320, out_fe

##### Training Preparation

In [7]:
train_dataset = CheXpertCaptionDataset(train_df, image_root, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-5)
scheduler = DDPMScheduler(num_train_timesteps=1000)

device = "cuda" if torch.cuda.is_available() else "cpu"
unet.to(device)
text_encoder.to(device)
vae.to(device)


AutoencoderKL(
  (encoder): Encoder(
    (conv_in): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down_blocks): ModuleList(
      (0): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0-1): 2 x ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (downsamplers): ModuleList(
          (0): Downsample2D(
            (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))
          )
        )
      )
      (1): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0): ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (c

#### Training Looop

In [8]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from transformers import CLIPTokenizer, CLIPTextModel

# ✅ Base output directory (no nesting)
base_output_dir = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/fine_tuning_stable_diffusion_model/model_checkpoints"
os.makedirs(base_output_dir, exist_ok=True)

start_epoch = 21
num_epochs = 30

# ✅ Load U-Net architecture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load models
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet").to(device)
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(device)
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to(device)


# ✅ Resume if checkpoint exists
resume_path = os.path.join(base_output_dir, f"unet_epoch_{start_epoch - 1}")
if os.path.exists(os.path.join(resume_path, "pytorch_model.bin")):
    unet.load_state_dict(torch.load(os.path.join(resume_path, "pytorch_model.bin")))
    print(f"Resumed from checkpoint: {resume_path}")

for epoch in range(start_epoch, num_epochs):
    unet.train()
    total_loss = 0.0

    for batch in train_loader:
        images = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        latents = vae.encode(images).latent_dist.sample() * 0.18215
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, 1000, (images.shape[0],), device=device).long()

        noisy_latents = scheduler.add_noise(latents, noise, timesteps)
        encoder_hidden_states = text_encoder(input_ids)[0]

        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}: Avg Loss = {avg_loss:.4f}")

    # ✅ Save U-Net to a flat folder (one per epoch)
    epoch_dir = os.path.join(base_output_dir, f"unet_epoch_{epoch + 1}")
    os.makedirs(epoch_dir, exist_ok=True)
    unet.save_pretrained(epoch_dir)
    print(f"Saved U-Net checkpoint to {epoch_dir}")

Epoch 22: Avg Loss = 0.1289
Saved U-Net checkpoint to /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/fine_tuning_stable_diffusion_model/model_checkpoints/unet_epoch_22
Epoch 23: Avg Loss = 0.1287
Saved U-Net checkpoint to /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/fine_tuning_stable_diffusion_model/model_checkpoints/unet_epoch_23
Epoch 24: Avg Loss = 0.1272
Saved U-Net checkpoint to /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/fine_tuning_stable_diffusion_model/model_checkpoints/unet_epoch_24
Epoch 25: Avg Loss = 0.1276
Saved U-Net checkpoint to /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/fine_tuning_stable_diffusion_model/model_checkpoints/unet_epoch_25
Epoch 26: Avg Loss = 0.1284
Saved U-Net checkpoint to /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/fine_tuning_stable_diffusion_model/model_checkpoints/unet_epoch_26
Epoch 27: Avg Loss = 0.1305
Saved U-Net checkpoint to /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/fine_tuning

KeyboardInterrupt: 

#### Load Pipeline for Inference

In [24]:
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, AutoencoderKL
from transformers import CLIPTextModel, CLIPTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Paths to your fine-tuned components
fine_tuned_unet_path = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/fine_tuning_stable_diffusion_model/model_checkpoints/unet_epoch_20"

# Load components
unet = UNet2DConditionModel.from_pretrained(fine_tuned_unet_path).to(device)
vae = AutoencoderKL.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="vae").to(device)
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to(device)
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")


demographics_training_validation_diseases_cnn/
densent121_Gen+Real_vs_real_training/
densnet12_real_vs_real_training/
Fine-Tuning_Our_Own_Models/
First_task_Visualizing_effects/
LORA_fine_tuning_stable_diffusion_model/
LORA_Model_Imagea_generation_with_prompt/
mixed_training_CNN_Model_Generated_Real/
old_CNN_model_training/
Pre_trained_models_for_images_generation/
resnet50_Gen+Real_vs_real_training/
resnet50_Gen+Real_vs_real_training_for_14_disease/
resnet50_real_vs_real_training/
resnet50_Real_vs_real_training_for_14_disease/
training_CNN_Models_On_generated_data/
training_CNN_Models_On_new_diffusion_generated_data/
training_CNN_Models_On_new_diffusion_generated_data_using_generated_imgs_As_training/
training_CNN_Models_On_new_diffusion_generated_data_with_fairness/
training_CNN_Models_On_real_and_valid_on_gen/
training_our_own_diffusion_model/
training_with_cfg/
Vision_Transformer_Training/


In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",  # Base model
    unet=unet,
    vae=vae,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
).to(device)


#### Generate Image from text prompt

In [ ]:
prompt = "Chest X-ray of a male patient of white race, aged 51 to 70, showing Enlarged Cardiomediastinum"
image = pipe(prompt, num_inference_steps=50).images[0]
image.save("xray_generated.png")

from IPython.display import display
display(image)